# Offline Mode (User-Managed vLLM) — Speculative Decoding Training

This notebook demonstrates how to train a custom **Eagle3 draft model** for speculative
decoding using the `OFFLINE` mode of `SpeculativeDecodingTrainer` from the Kubeflow SDK
on Red Hat OpenShift AI.

## What is OFFLINE Mode?

The `OFFLINE` mode connects to a **user-managed external vLLM server** to extract hidden
states from the verifier model, then trains the Eagle3 draft model — all within a single
job. Unlike `ONLINE` mode, the SDK does **not** deploy a vLLM sidecar. You provide a
`vllm_endpoint` pointing to your own vLLM instance.

This is useful when you already have a vLLM deployment running (e.g., as an OpenShift AI
model serving instance) and want to reuse it for hidden state extraction.

## How It Works

1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC
3. Training runs immediately after extraction completes — all within the same job

## Speculative Decoding Overview

Large language models generate tokens one at a time, and each token requires reading the
entire model from GPU memory — making inference **memory-bound**. Speculative decoding
exploits this: a small, fast **draft model** (~1 GB) guesses the next several tokens,
then the large **verifier model** checks all guesses in a single forward pass. The output
is mathematically identical to normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate
layers of the verifier (not just the final logits), giving it richer context for more
accurate predictions.

## Dataset

This example uses the `magpie` built-in dataset (Magpie-format conversation dataset).

## Hardware Requirements

| Component | GPU | CPU | Memory | Notes |
|-----------|-----|-----|--------|-------|
| Training container | 2× NVIDIA L40S / A100 | 4 cores | 64Gi | Runs Eagle3 draft model training |
| External vLLM server | 1× NVIDIA L40S / A100 | 4 cores | 96Gi | User-managed; must be running before job submission |

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth and create the API client
from kubernetes import client

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

Provide your OpenShift API server URL, authentication token, and HuggingFace token.
Update `PVC_NAME` to match the name of your shared RWX PersistentVolumeClaim.

In [ ]:
# ============================================================================
# CLUSTER AUTHENTICATION
# ============================================================================
# Replace these with your OpenShift cluster API server URL and bearer token.
# In OpenShift AI workbenches, these may be available as environment variables
# (OPENSHIFT_API_URL, NOTEBOOK_USER_TOKEN) — but for clarity we set them explicitly.
api_server = "<REPLACE WITH OPENSHIFT SERVER>"
token = "<REPLACE WITH OPENSHIFT TOKEN>"

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# ============================================================================
# KUBERNETES CLIENT CONFIGURATION
# ============================================================================
# The Configuration object holds the API server URL, auth token, and TLS settings.
# This is passed to the ApiClient, which the TrainerClient uses for all cluster operations.
configuration = client.Configuration()
configuration.host = api_server

# Uncomment if your cluster API server uses a self-signed certificate or an untrusted CA
# configuration.verify_ssl = False

configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = client.ApiClient(configuration)

# ============================================================================
# PVC CONFIGURATION
# ============================================================================
# PVC_NAME must match the ReadWriteMany (RWX) PVC attached to your workbench.
# The notebook sees it at /opt/app-root/src/<pvc-name> (OpenShift AI convention).
# Training pods see it at /mnt/kubeflow-checkpoints (SDK constant CHECKPOINT_MOUNT_PATH).
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        f"Warning: Expected workbench PVC mount not found at: {NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME above.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT AND CLUSTER TRAINING RUNTIME
# ============================================================================
# Create the TrainerClient — the main SDK entry point for submitting and managing TrainJobs.
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(
        client_configuration=api_client.configuration
    )
)

# ClusterTrainingRuntime (CTR) for OFFLINE mode.
# OFFLINE mode does not use a managed vLLM sidecar, so only the model optimization CTR is needed.
MODEL_OPT_CTR = "speculator-model-opt-cuda"  # Training only, no vLLM sidecar

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found" if MODEL_OPT_CTR in available_runtimes else "WARNING: not found on cluster"
)
print(f"CTR '{MODEL_OPT_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## Configuration

The following constants configure the training run. The verifier model is
[Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), a 36-layer transformer,
pre-downloaded to the shared PVC. All storage paths use **PVC URIs**
(`pvc://<pvc-name>/<path>`), which the SDK resolves to container mount paths internally.

When the verifier model is a PVC URI, the SDK cannot read the model config to
auto-detect layer IDs, so `target_layer_ids` must be provided explicitly via
`SpeculatorConfig`.

You must also set `VLLM_ENDPOINT` to point to your external vLLM server serving
the same verifier model.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
VERIFIER_MODEL = "Qwen/Qwen3-8B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-8B has 36 transformer layers (indexed 1-36).
# Layers chosen: early (3), mid (18), late (33), and final (36) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [3, 18, 33, 36]

# GPU, CPU, and memory allocations for the training container.
# 2 GPUs enable data-parallel training; 64Gi memory holds model weights + optimizer state.
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 2,
    "cpu": "4",
    "memory": "64Gi",
}

# URL of your externally managed vLLM server.
# This must be a running vLLM instance serving the same verifier model (Qwen3-8B).
# The /v1 path exposes the OpenAI-compatible API that the SDK calls for extraction.
VLLM_ENDPOINT = "http://vllm-svc.speculative-decoding.svc.cluster.local:8000/v1"

# Training hyperparameters
EPOCHS = 3  # Number of full passes over the training data
LEARNING_RATE = 1e-4  # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for both extraction and training
MAX_SAMPLES = 5000  # Cap on the number of dataset samples to process

print("OFFLINE Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  vLLM endpoint:     {VLLM_ENDPOINT}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## Offline Mode (User-Managed vLLM)

The `OFFLINE` mode extracts hidden states via a user-managed external vLLM server,
then trains the draft model in a single job. This is useful when you already have a
vLLM deployment running (e.g., as an OpenShift AI model serving instance) and want to
reuse it for hidden state extraction instead of having the SDK deploy a sidecar.

**How it works:**
1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC at `hidden_states_path`
3. Training runs immediately after extraction completes — all within the same job

**Key differences from other modes:**
- You must provide `vllm_endpoint` pointing to your external vLLM server
- The SDK does not deploy a vLLM sidecar — `vllm_resources` is not used
- Both `training_resources` (for the training container) and `vllm_endpoint`
  (for extraction) are required

We use the `magpie` built-in dataset for this example.

In [ ]:
OFFLINE_JOB = f"eagle3-offline-{RUN_NAME}"
OFFLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-offline"

# Configure the OFFLINE trainer.
# OFFLINE mode connects to an external vLLM endpoint for extraction, then trains.
# Both steps happen within the same job — extraction first, training second.
# Unlike DATA_ONLY + TRAIN_ONLY, this is a single-job workflow.
# Unlike ONLINE, the SDK does NOT deploy a vLLM sidecar — you manage it yourself.
offline_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.OFFLINE,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    dataset_name="magpie",  # Built-in Magpie conversation dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_endpoint=VLLM_ENDPOINT,  # External vLLM server URL
    hidden_states_path=f"{OFFLINE_OUTPUT}/hidden_states",  # Where extracted states are saved
    training_resources=TRAINING_RESOURCES,  # Resources for the training container
    regenerate_responses=True,  # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=OFFLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        resume_from_checkpoint=True,  # Resume from the latest checkpoint if one exists
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("OFFLINE Configuration:")
print(f"  Job name:         {OFFLINE_JOB}")
print(f"  Mode:             {offline_trainer.mode.value}")
print(f"  Verifier (PVC):   {offline_trainer.verifier_model}")
print(f"  vLLM endpoint:    {offline_trainer.vllm_endpoint}")
print(f"  Dataset:          {offline_trainer.dataset_name}")
print(f"  Target layers:    {offline_trainer.config.target_layer_ids}")
print(f"  Hidden states:    {offline_trainer.hidden_states_path}")
print(f"  Output dir:       {offline_trainer.output_dir}")

In [ ]:
# Submit the OFFLINE TrainJob to the cluster.
# Uses MODEL_OPT_CTR — no SDK-managed vLLM sidecar (the external endpoint handles extraction).
trainer_client.train(
    options=[Name(name=OFFLINE_JOB)],
    trainer=offline_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"OFFLINE job submitted: {OFFLINE_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={OFFLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the OFFLINE job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(OFFLINE_JOB)

## Cleanup

Delete the TrainJob when you are done.

In [ ]:
# Delete completed TrainJob to free cluster resources (pods, volumes, etc.).
# Note: Deleting a job does NOT delete the output data on the PVC —
# checkpoints and hidden states remain available for future use.

# trainer_client.delete_job(OFFLINE_JOB)
# print("OFFLINE TrainJob deleted.")

## Summary

This notebook demonstrated **OFFLINE** mode — extracting hidden states via an external
vLLM endpoint and training an Eagle3 draft model in a single job using the `magpie`
dataset.

OFFLINE mode is ideal when you already have a vLLM deployment running and want to
reuse it for hidden state extraction instead of having the SDK deploy a managed sidecar.

### Key Takeaways

- `vllm_endpoint` points to your external vLLM server — the SDK does not deploy a sidecar
- Both extraction and training happen in a single job
- The external vLLM server must be serving the same verifier model used in training
- All storage paths use **PVC URIs** (`pvc://<pvc-name>/<path>`)

### Next Steps

- Deploy the trained draft model with vLLM for speculative decoding inference
- Adjust `epochs`, `lr`, and `max_samples` to tune draft model quality
- Try [DATA_ONLY + TRAIN_ONLY](../data-only/) to separate extraction from training
  and iterate on hyperparameters without re-running extraction
- Try [ONLINE](../online/) mode for the fully managed alternative where the SDK
  deploys the vLLM sidecar automatically